In [15]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [11]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env import BalanceBotEnv
from rl.ppo_trainer import evaluate, train, PPOConfig

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 1   # Single environment for now (so we can watch it train)

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = 30_000,     # Total simulation steps across all envs and iterations
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 10,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering
)

In [5]:
# Build the environment
env = BalanceBotEnv(
    mjcf_path    = MJCF_PATH,
    render_mode  = "human",
)

# Wrap in RecordEpisodeStatistics so we can log episodic returns
env = gym.wrappers.RecordEpisodeStatistics(env)

# Wrap in SyncVectorEnv so it's compatible with train()
envs = gym.vector.SyncVectorEnv([lambda: env])

In [8]:
# Choo choo train!
result = train(ppo_config, envs=envs)

global_step=124, episodic_return=81.96
global_step=315, episodic_return=155.68
global_step=505, episodic_return=150.85
global_step=637, episodic_return=91.41
global_step=895, episodic_return=191.83
global_step=1443, episodic_return=509.73
global_step=1569, episodic_return=86.40
global_step=1746, episodic_return=121.44
global_step=1843, episodic_return=64.89
global_step=1986, episodic_return=91.15
global_step=2227, episodic_return=184.34
global_step=2400, episodic_return=119.53
global_step=2644, episodic_return=194.60
global_step=2911, episodic_return=216.12
global_step=3045, episodic_return=99.88
global_step=3222, episodic_return=140.32
global_step=3438, episodic_return=169.46
global_step=3566, episodic_return=88.11
global_step=3693, episodic_return=87.95
global_step=3928, episodic_return=180.31
global_step=4068, episodic_return=97.85
global_step=4389, episodic_return=275.98
global_step=4550, episodic_return=111.58
global_step=4721, episodic_return=106.41
global_step=4883, episodic_ret

In [9]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1777733939/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1777733939/balance-bot-ppo_final.cleanrl_model
Best mean return: 270.31
Loaded best model (mean_return=270.31)


In [17]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.002)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=10, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 279.92


In [7]:
# Close the environments
envs.close()

In [13]:
# %%%TEST: Export training metrics as CSV. Make this into separate Python script?

import csv
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from pathlib import Path

# Point to a specific run directory
run_path = Path("/workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207")

# Load the event data
ea = EventAccumulator(run_path)
ea.Reload()

# See what metrics are available
all_tags = ea.Tags()['scalars']
print(f"Available metrics: {all_tags}")

# Split into training and eval metrics
train_tags = [t for t in all_tags if not t.startswith('eval/')]
eval_tags  = [t for t in all_tags if t.startswith('eval/')]

def export_csv(tags, output_path):
    """Export a list of scalar tags to a single CSV file."""
    if not tags:
        print(f"No tags to export for {output_path}")
        return

    # Collect all data: {tag: [(step, value), ...]}
    data = {}
    all_steps = set()
    for tag in tags:
        events = ea.Scalars(tag)
        data[tag] = {e.step: e.value for e in events}
        all_steps.update(data[tag].keys())

    # Write CSV with one column per metric, rows indexed by step
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)

        # Header row
        writer.writerow(['step'] + tags)

        # One row per step, fill missing values with empty string
        for step in sorted(all_steps):
            row = [step] + [data[tag].get(step, '') for tag in tags]
            writer.writerow(row)

    print(f"Exported {len(tags)} metrics to {output_path}")

export_csv(train_tags, run_path / "training_metrics.csv")
export_csv(eval_tags,  run_path / "eval_metrics.csv")

Available metrics: ['charts/learning_rate', 'losses/value_loss', 'losses/policy_loss', 'losses/entropy', 'losses/old_approx_kl', 'losses/approx_kl', 'losses/clipfrac', 'losses/explained_variance', 'charts/SPS', 'eval/episodic_return']
Exported 9 metrics to /workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207/training_metrics.csv
Exported 1 metrics to /workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207/eval_metrics.csv
